In [1]:

import pandas as pd
from pathlib import Path

# File paths
data_dir = Path("../data/processed")

file_2001 = data_dir / "census_2001_cleaned.csv"
file_2011 = data_dir / "census_2011_cleaned.csv"

# Load datasets
df_2001 = pd.read_csv(file_2001)
df_2011 = pd.read_csv(file_2011)

# Add census year
df_2001["Census_Year"] = 2001
df_2011["Census_Year"] = 2011

print("2001 shape:", df_2001.shape)
print("2011 shape:", df_2011.shape)

print("\n2001 columns:")
print(df_2001.columns.tolist())

print("\n2011 columns:")
print(df_2011.columns.tolist())


2001 shape: (1547, 35)
2011 shape: (1715, 35)

2001 columns:
['Table_Code', 'State_Code', 'District_Code', 'Tehsil_Code', 'Town_Code', 'Area_Name', 'Area_Type', 'Total_Persons', 'Total_Males', 'Total_Females', 'Hindu_Persons', 'Hindu_Males', 'Hindu_Females', 'Muslim_Persons', 'Muslim_Males', 'Muslim_Females', 'Christian_Persons', 'Christian_Males', 'Christian_Females', 'Sikh_Persons', 'Sikh_Males', 'Sikh_Females', 'Buddhist_Persons', 'Buddhist_Males', 'Buddhist_Females', 'Jain_Persons', 'Jain_Males', 'Jain_Females', 'Other_Religions_Persons', 'Other_Religions_Males', 'Other_Religions_Females', 'Religion_Not_Stated_Persons', 'Religion_Not_Stated_Males', 'Religion_Not_Stated_Females', 'Census_Year']

2011 columns:
['Table_Code', 'State_Code', 'District_Code', 'Tehsil_Code', 'Town_Code', 'Area_Name', 'Area_Type', 'Total_Persons', 'Total_Males', 'Total_Females', 'Hindu_Persons', 'Hindu_Males', 'Hindu_Females', 'Muslim_Persons', 'Muslim_Males', 'Muslim_Females', 'Christian_Persons', 'Christ

In [2]:

# Select Maharashtra state-level rows
state_2001 = df_2001[
    df_2001["Area_Name"].astype(str).str.strip().str.upper()
    == "MAHARASHTRA"
].copy()

state_2011 = df_2011[
    df_2011["Area_Name"].astype(str).str.strip().str.upper()
    == "STATE - MAHARASHTRA"
].copy()

print("2001 state rows:", len(state_2001))
print("2011 state rows:", len(state_2011))

print("\n2001:")
print(state_2001[["Area_Name", "Area_Type", "Total_Persons"]].to_string(index=False))

print("\n2011:")
print(state_2011[["Area_Name", "Area_Type", "Total_Persons"]].to_string(index=False))


2001 state rows: 3
2011 state rows: 3

2001:
  Area_Name Area_Type  Total_Persons
MAHARASHTRA     Total       96878627
MAHARASHTRA     Rural       55777647
MAHARASHTRA     Urban       41100980

2011:
          Area_Name Area_Type  Total_Persons
State - MAHARASHTRA     Total      112374333
State - MAHARASHTRA     Rural       61556074
State - MAHARASHTRA     Urban       50818259


In [3]:

# Create a comparison table
comparison = pd.DataFrame({
    "Area_Type": ["Total", "Rural", "Urban"],
    "Population_2001": [
        state_2001.loc[state_2001["Area_Type"] == area, "Total_Persons"].iloc[0]
        for area in ["Total", "Rural", "Urban"]
    ],
    "Population_2011": [
        state_2011.loc[state_2011["Area_Type"] == area, "Total_Persons"].iloc[0]
        for area in ["Total", "Rural", "Urban"]
    ]
})

comparison["Population_Change"] = (
    comparison["Population_2011"] - comparison["Population_2001"]
)

comparison["Growth_Percent"] = (
    comparison["Population_Change"]
    / comparison["Population_2001"] * 100
).round(2)

print(comparison.to_string(index=False))

# Save comparison
comparison.to_csv(
    data_dir / "maharashtra_population_comparison.csv",
    index=False
)

print("\nComparison CSV saved successfully!")


Area_Type  Population_2001  Population_2011  Population_Change  Growth_Percent
    Total         96878627        112374333           15495706           15.99
    Rural         55777647         61556074            5778427           10.36
    Urban         41100980         50818259            9717279           23.64

Comparison CSV saved successfully!


In [4]:

import pandas as pd
from pathlib import Path

data_dir = Path("../data/processed")

df_2001 = pd.read_csv(data_dir / "census_2001_cleaned.csv")
df_2011 = pd.read_csv(data_dir / "census_2011_cleaned.csv")

# Select state-level Total rows
state_2001 = df_2001[
    (df_2001["Area_Name"].astype(str).str.strip().str.upper() == "MAHARASHTRA") &
    (df_2001["Area_Type"].astype(str).str.strip().str.upper() == "TOTAL")
]

state_2011 = df_2011[
    (df_2011["Area_Name"].astype(str).str.strip().str.upper() == "STATE - MAHARASHTRA") &
    (df_2011["Area_Type"].astype(str).str.strip().str.upper() == "TOTAL")
]

if len(state_2001) != 1 or len(state_2011) != 1:
    raise ValueError(
        f"Expected one state Total row per year. "
        f"Found 2001={len(state_2001)}, 2011={len(state_2011)}"
    )

state_2001 = state_2001.iloc[0]
state_2011 = state_2011.iloc[0]

# Religion population columns
religions = {
    "Hindu": ("Hindu_Persons", "Hindu_Persons"),
    "Muslim": ("Muslim_Persons", "Muslim_Persons"),
    "Christian": ("Christian_Persons", "Christian_Persons"),
    "Sikh": ("Sikh_Persons", "Sikh_Persons"),
    "Buddhist": ("Buddhist_Persons", "Buddhist_Persons"),
    "Jain": ("Jain_Persons", "Jain_Persons"),
    "Other Religions": (
        "Other_Religions_Persons", "Other_Religions_Persons"
    ),
    "Religion Not Stated": (
        "Religion_Not_Stated_Persons",
        "Religion_Not_Stated_Persons"
    )
}

rows = []

for religion, (col_2001, col_2011) in religions.items():
    pop_2001 = int(state_2001[col_2001])
    pop_2011 = int(state_2011[col_2011])

    rows.append({
        "Religion": religion,
        "Population_2001": pop_2001,
        "Population_2011": pop_2011,
        "Population_Change": pop_2011 - pop_2001,
        "Growth_Percent": round(
            (pop_2011 - pop_2001) / pop_2001 * 100, 2
        ) if pop_2001 else None,
        "Share_2001_Percent": round(
            pop_2001 / int(state_2001["Total_Persons"]) * 100, 2
        ),
        "Share_2011_Percent": round(
            pop_2011 / int(state_2011["Total_Persons"]) * 100, 2
        )
    })

religion_comparison = pd.DataFrame(rows)

output_file = data_dir / "religion_comparison_2001_2011.csv"
religion_comparison.to_csv(output_file, index=False)

print(religion_comparison.to_string(index=False))
print(f"\nSaved successfully: {output_file.resolve()}")


           Religion  Population_2001  Population_2011  Population_Change  Growth_Percent  Share_2001_Percent  Share_2011_Percent
              Hindu         77859385         89703057           11843672           15.21               80.37               79.83
             Muslim         10270485         12971152            2700667           26.30               10.60               11.54
          Christian          1058313          1080073              21760            2.06                1.09                0.96
               Sikh           215337           223247               7910            3.67                0.22                0.20
           Buddhist          5838710          6531200             692490           11.86                6.03                5.81
               Jain          1301843          1400349              98506            7.57                1.34                1.25
    Other Religions           236841           178965             -57876          -24.44         